[![Open In Colab](./colab-badge.png)](https://colab.research.google.com/github/subhacom/moose-notebooks/blob/main/Dynamical_system_with_NaP.ipynb) [![Binder](./binder_logo.png)](https://mybinder.org/v2/gh/subhacom/moose-notebooks/HEAD?labpath=Dynamical_system_with_NaP.ipynb)

This tutorial introduces a 1-dimensional dynamical system following the book "Dynamical Systems in Neuroscience" by Eugene Izhikevich.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import moose

In [ ]:
model_path = '/model'
data_path = '/data'

if moose.exists(model_path):
    moose.delete(model_path)
if moose.exists(data_path):
    moose.delete(data_path)

model = moose.Neutral(model_path)
data = moose.Neutral(data_path)

We will model this simplified system:

$C\frac{dV}{dt} = I - (V - E_{m})/R_{m} - \bar{G}_{Na} m_{\infty}(V)(V - E_{Na})$


Here the $Na^{+}$ channel dynamics is considered to be very fast, so that $m(V)$ reaches steady state value $m_{\infty}(V)$ instantaneously.

This equation without the $Na^{+}$ current is already simulated by the `Compartment` class. We just need to add an $Na^{+}$ channel to simulate this model.

In [ ]:
comp = moose.Compartment(f'{model_path}/soma')
na_chan = moose.HHChannel(f'{comp.path}/NaP')
moose.connect(comp, 'channel', na_chan, 'channel')

In [ ]:
vm_tab = moose.Table(f'{data.path}/Vm')
moose.connect(vm_tab, 'requestOut', comp, 'getVm')

We will use a simple sigmoid for $m_{\infty}(V)$: $m_{\infty}(V) = \frac{1}{1 + \exp{\left(\frac{ V_{1/2} - V}{k}\right) }}$

where $V_{1/2}$ is the midpoint of the sigmoid, and $k$ is the slope.

We are making this model based on Chapter 3, section 3.1 of the book "Dynamical Systems in Neuroscience" by Eugene Izhikevich. The parameters are:

$C_{m} = 10 \mu F$

$\bar{G}_{Na} = 74 mS$

$E_{Na} = 60 mV$

$g_{L} = 1/R_{m} = 19 mS$

$E_{L} = E_{m} = -67 mV$

$V_{1/2} = 1.5 mV$

$k = 16 mV$

In [ ]:
na_chan.Xpower = 1
na_chan.instant = 1  # this makes X gate instantaneous
mgate = moose.element(na_chan.path + '/gateX')
mgate.infExpr = '1/(1 + exp((1.5e-3 - v)/16e-3))'
mgate.tauExpr = '1'
mgate.min = -120e-3   # Minimum voltage
mgate.max = 40e-3     # Maximum voltage
mgate.divs = 3000     # Granularity of the lookup table for minf

In [ ]:
comp.Cm = 10e-6
comp.Rm = 1/19e-3
comp.Em = -67e-3
na_chan.Gbar = 74e-3
na_chan.Ek = 60e-3


Now we will simulate the time course of the voltage under different initial membrane potentials:

In [ ]:
simtime = 15e-3

comp.inject = 0.0
for v in np.arange(-65e-3, 40e-3, 5e-3):
    comp.initVm = v
    moose.reinit()
    moose.start(simtime)
    t = np.linspace(0, simtime, len(vm_tab.vector))
    plt.plot(t*1e3, vm_tab.vector*1e3)

Visibly, the neuron goes to one of two voltages asymptotically, depending on the initial membrane potential.

Now let us change the constant current injection to $I = 60 \mu A$ rerun the simulation.

In [ ]:
simtime = 15e-3

comp.inject = 60e-6   # 60 uA constant current injection
for v in np.arange(-65e-3, 40e-3, 10e-3):
    comp.initVm = v
    moose.reinit()
    moose.start(simtime)
    t = np.linspace(0, simtime, len(vm_tab.vector))
    plt.plot(t*1e3, vm_tab.vector*1e3)    

Now there is only one voltage towards which the membrane potential goes asymptotically.